**day2_sbert_faiss**

In [1]:
!pip install sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.7 MB/s eta 0:00:00


**Block 2 — Load the model and understand what it does**


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Test it on one sentence first
sentence = "Machine learning models learn patterns from data."
embedding = model.encode(sentence)

print("Embedding shape:", embedding.shape)
print("First 5 numbers:", embedding[:5])
# ```

# You will see shape `(384,)` — meaning every sentence becomes a list of 384 numbers.

# **Add text cell:**
# ```
# SBERT converts any sentence into a vector of 384 numbers.
# Similar sentences produce similar vectors — this is how semantic search works.
# Unlike BERT, SBERT is optimised specifically for sentence similarity.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (384,)
First 5 numbers: [-0.05634136 -0.05581019  0.04128261  0.0302838   0.03201018]


**Block 3 — See why it's called SEMANTIC search**

In [4]:
from sentence_transformers import util

# These mean the same thing but use different words
s1 = "I love programming in Python"
s2 = "Coding in Python is my passion"
s3 = "I enjoy eating pizza"

e1 = model.encode(s1)
e2 = model.encode(s2)
e3 = model.encode(s3)

sim_12 = util.cos_sim(e1, e2).item()
sim_13 = util.cos_sim(e1, e3).item()

print(f"s1 vs s2 (same meaning): {round(sim_12, 3)}")
print(f"s1 vs s3 (different meaning): {round(sim_13, 3)}")
# ```

# You will see s1 vs s2 score is much higher than s1 vs s3.

# **Add text cell:**
# ```
# Cosine similarity measures the angle between two vectors.
# Score close to 1.0 = very similar meaning.
# Score close to 0.0 = unrelated.
# This is why it works even when the exact words are different.

s1 vs s2 (same meaning): 0.893
s1 vs s3 (different meaning): 0.285


**Block 4 — Build a FAISS index (this is the core)**

In [5]:
import faiss
import numpy as np

# Your corpus — 15 sentences
sentences = [
    "Machine learning models require large amounts of data.",
    "Python is the most popular language for data science.",
    "Neural networks are inspired by the human brain.",
    "FAISS is a library for fast similarity search.",
    "Transformers revolutionized natural language processing.",
    "Gradient descent optimizes model parameters iteratively.",
    "BERT is a bidirectional encoder representation model.",
    "GPT generates text by predicting the next token.",
    "Embeddings represent words and sentences as dense vectors.",
    "Fine-tuning adapts pretrained models to specific tasks.",
    "Attention mechanism helps models focus on relevant tokens.",
    "Overfitting happens when a model memorizes training data.",
    "LangChain is a framework for building LLM applications.",
    "RAG combines retrieval with language model generation.",
    "Sentence-BERT produces high quality sentence embeddings.",
]

# Embed all sentences
embeddings = model.encode(sentences)
embeddings = np.array(embeddings).astype('float32')

# Build FAISS index
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Index built. Total sentences stored: {index.ntotal}")

Index built. Total sentences stored: 15


**Block 5 — Search the index**

In [6]:
def search(query, top_k=3):
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k)

    print(f"Query: '{query}'")
    print(f"\nTop {top_k} results:")
    for i, idx in enumerate(indices[0]):
        print(f"  {i+1}. {sentences[idx]}")
        print(f"     Distance: {round(distances[0][i], 3)}")
    print()

# Run 3 different searches
search("how do transformers work in NLP")
search("what is overfitting in machine learning")
search("how to build apps with language models")

Query: 'how do transformers work in NLP'

Top 3 results:
  1. Transformers revolutionized natural language processing.
     Distance: 0.593999981880188
  2. BERT is a bidirectional encoder representation model.
     Distance: 1.1380000114440918
  3. RAG combines retrieval with language model generation.
     Distance: 1.2400000095367432

Query: 'what is overfitting in machine learning'

Top 3 results:
  1. Overfitting happens when a model memorizes training data.
     Distance: 0.47200000286102295
  2. Machine learning models require large amounts of data.
     Distance: 0.9580000042915344
  3. Gradient descent optimizes model parameters iteratively.
     Distance: 1.347000002861023

Query: 'how to build apps with language models'

Top 3 results:
  1. LangChain is a framework for building LLM applications.
     Distance: 1.0839999914169312
  2. RAG combines retrieval with language model generation.
     Distance: 1.1380000114440918
  3. Fine-tuning adapts pretrained models to specific 

Block 6 — Your own test


In [9]:
# Write your OWN query here — anything related to ML/NLP
# See what comes back

search("why model fails")

# Then answer in a comment:
# Q: What is FAISS actually doing under the hood?
# A:

Query: 'why model fails'

Top 3 results:
  1. Overfitting happens when a model memorizes training data.
     Distance: 1.4210000038146973
  2. Gradient descent optimizes model parameters iteratively.
     Distance: 1.472000002861023
  3. Machine learning models require large amounts of data.
     Distance: 1.4919999837875366

